# **Detecção de eixos** de caminhões para inferência de carga máxima - utilizando YOLOv8

### Problema
Imprecisão na emissão de alertas automáticos para auditores sobre veículos que poderiam levantar alertas de incongruências de carga em relação às notas fiscais emitidas. 

### Proposta
Os eixos de um caminhão indicam, juntamente com outros fatores, qual é a carga máxima desde veículo. A ideia é detectar essa informação e, de acordo com regras pré-estabelecidas, inferir uma possível carga máxima para um dado veículo (do tipo **caminhão**) analisado.


### Pipeline

1. Detecção de pneus
    
    1.1. Análise e pré-processamento da database (imagens e informações sobre caminhões registrados pelo sistema de monitoramento de veículos de carga da SEFAZ-PB)

    1.2. Anotação das imagens (820 imagens de caminhão)

    1.3. Organização das anotações e imagens que serão submetidas ao modelo *Deep Learning* para *Visão Computacional*  **YOLOv8** 

    1.4. Treinamento do modelo utilizando os conjuntos de treinamento e teste

    1.5. Avaliação do modelo de detecção de eixos

    1.6. Utilização na prática

2. Clusterização de **Pneus** em **Eixos**

    2.1. Estudo de clusterização

    2.2. Implementação da *função de clusterização* elaborada

    2.3. Testes práticos

# Detecção de Eixos

## Análise e Pré-processamento

In [27]:
# para recarregar automaticamente os módulos editados
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Anotação das imagens

## Organização dos dados (Padrão YOLOv8)

O modelo **YOLOv8** pede uma estrutura organizada de uma forma específica para operar de forma previsível. Utilizaremos a estrutura a seguir:

```
data/
    train/
        /images
        /labels
    val/
        /images
        /labels
    test/
        /images
        /labels

custom_dataset.yaml
```
---

Para chegar a esta organização, é necessário:
1.   Dividir o dataset em treino, validação e teste
2.   Distribuir nas pastas

In [30]:
# função que faz os dois passos 
from utils import split_train_val

split_train_val(
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1,
    random_seed=42
)

{'total': 820, 'train': 574, 'val': 164, 'test': 82}

## Treinamento

Importação da bilbioteca que contém a classe YOLO

In [ ]:
%pip install ultralytics

In [38]:
from ultralytics import YOLO, settings

Modelos disponíveis para download na Ultralytics disponíveis na documentação: [yolov8](https://docs.ultralytics.com/models/yolov8/#key-features-of-yolov8) 

Para ajustar as configurações do ultralytics, consultar documentação disponível em [ultralytics](https://docs.ultralytics.com/quickstart/#modifying-settings)

In [ ]:
settings

In [48]:
settings.update({
    'datasets_dir': '/home/aria-unimed/axle-detection-herick/',
})

### Iniciar arquitetura

In [ ]:
# arquitetura padrão, por hora
model = YOLO('yolov8n')
    
# Em breve, alterar arquitetura para buscar ajustar ao problema específico, 
# e não usar a arquitetura genérica

Argumentos (hiperparâmentros): [ultralytics-hyperparams](https://docs.ultralytics.com/modes/train/#musgd-optimizer)

Parâmetros de data augmentation: [ultralytics-data-augmentation](https://docs.ultralytics.com/modes/train/#augmentation-settings-and-hyperparameters)

In [52]:
model.train(
    data='custom_dataset.yaml',
    epochs=30,
    patience=5, # número de épocas sem melhoria para parar o treinamento
    batch=-1, # usar o batch size padrão, que é o melhor para a arquitetura escolhida
    imgsz=640, # tamanho da imagem de entrada
    workers=8, # número de threads utilizadas
    pretrained=True, # usar pesos pré-treinados para acelerar o treinamento
    resume=False, # não retomar de um treinamento anterior
    box=7.5, # peso para a perda de caixa delimitadora, para dar mais importância à localização precisa dos eixos
    cls=0.5, # peso para a perda de classificação, para dar mais importância à identificação correta dos eixos
    dfl=1.5, # peso para a perda de distribuição de distância, para melhorar a precisão da localização dos eixos
    val=True, # realizar validação durante o treinamento para monitorar o desempenho do modelo
    # augmentations
)

Ultralytics 8.4.33 🚀 Python-3.10.12 torch-2.5.1+cu124 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 7754MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=custom_dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, per

/home/aria-unimed/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
AutoBatch: Computing optimal batch size for imgsz=640 at 60.0% CUDA memory utilization.
AutoBatch: CUDA:0 (NVIDIA GeForce RTX 2060 SUPER) 7.57G total, 0.12G reserved, 0.07G allocated, 7.38G free
      Params      GFLOPs  GPU_mem (GB)  forward (ms) backward (ms)                   input                  output
     3011043       8.194         0.428         24.71           nan        (1, 3, 640, 640)                    list
     3011043       16.39         0.828         14.59           nan        (2, 3, 640, 640)                    list
     3011043       32.78         1.332         13.14           nan        (4, 3, 640, 640)                    list
     3011043       65.55         2.506         17.14           nan        (8, 3, 640, 640)                    list
     3011043    

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8261d94760>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

## Avaliação

## Uso prático